# Performance

Owner: Hiếu. Testbed: **Google Colab, CPU-only**. Numbers come from `data/m1/benchmark_summary.csv` and `data/m1/resource_timeseries.csv`.

:::{warning}
The committed CSVs are **placeholder data**, not final measurements. Final results must cover all 20,000 assigned IDs and their versions. The audited 200-ID Dang smoke is exploratory and retains its original measurement scope. New full-session measurements require a new run ID.
:::

The [operational crawl and benchmark notebook](arxiv_nhom1_crawl_benchmark.ipynb) is rendered separately without execution. See [methodology](01-methodology.md) for timing boundaries, observed sampled resource peaks and interrupted-attempt limitations. Retry/backoff is already included in wall time; never add it again. A complete corpus does not guarantee complete timing.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("../data/m1")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

bench = pd.read_csv(DATA / "benchmark_summary.csv")
ts = pd.read_csv(DATA / "resource_timeseries.csv")

## Summary

In [ ]:
bench.set_index("metric")

## RAM and disk over time

For new measurement runs, `rss_total_mb` includes the runner/notebook kernel baseline and all descendants, throughout restore, crawling and final corpus sync. Disk covers local data/work and telemetry staging. Chunk-only legacy CSVs have a narrower scope. Peaks are sampled observations; environment setup and final evidence re-export/download are excluded.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
groups = ts.groupby("session_id", sort=False) if "session_id" in ts else [("legacy", ts)]
for _, part in groups:
    ax1.plot(part.t_sec / 60, part.rss_total_mb)
    ax2.plot(part.t_sec / 60, part.disk_mb, color="tab:orange")
ax1.set_ylabel("RAM (MB)")
ax2.set_ylabel("Disk (MB)")
ax2.set_xlabel("Concatenated observed session time (minutes; offline gaps excluded)")
fig.tight_layout()
fig.savefig(FIG / "m1_resources.svg")

In [ ]:
# The authoritative time-weighted RAM mean is exported by the sampler.
# Do not average concatenated samples arithmetically across unequal sessions.
audit_path = DATA / "measurement_audit.json"
if audit_path.exists():
    import json
    print(json.dumps(json.loads(audit_path.read_text()), indent=2))
else:
    print("Legacy/placeholder CSVs: no full-session measurement audit supplied.")


## Audit evidence

The benchmark ZIP retains `/usr/bin/time -v` logs, chunk profiles, full-session sample segments, per-paper attempt timings, manifests and QA. Publish the final run's evidence link with these results before submission. `/usr/bin/time` wraps the chunk profiler and does not report simultaneous total process-tree RSS.

Interrupted sessions expose an observed wall-time lower bound and omit exact end-to-end/average-time rows. Do not treat missing intervals as zero. Per-paper lifecycle latencies include batch queue waiting and overlap; use end-to-end wall divided by required IDs for throughput.
